# YOLOv1 — validation inference demo (50-epoch self-defined model)

**Add Input:**
1. [PASCAL VOC 2007](https://www.kaggle.com/datasets/stpeteishii/pascal-voc-2007-dataset/data).
2. Your **full-training output**, containing `checkpoints/best.pt`, or its training ZIP. The archive name may contain `smoke`; its saved config determines which run it really is.

Enable Internet for cloning code. One GPU is sufficient; T4 ×2 is also fine (inference uses GPU 0).
The notebook runs fresh inference over the complete official **val** split, computes VOC2007 mAP and threshold metrics, selects up to **3 illustrative images**, and exports a ZIP without model weights or the dataset.

This is an implementation demo, not a reproduction of the original paper benchmark. Examples are deliberately selected using GT after inference; full-val metrics remain separate and unchanged.


In [ ]:
REPO_URL = "https://github.com/Dacekey/YOLOv1_PersonalTask_DPL302m.git"
REPO_REF = "main"  # or the full commit containing inference_demo.py after pushing
DATASET_DIR = ""  # auto-find VOC2007 under /kaggle/input; use an absolute path if ambiguous
CHECKPOINT_PATH = ""  # auto-find best.pt/training ZIP; or /kaggle/input/.../best.pt or .../training.zip
RUN_NAME = "yolov1_voc2007_val_demo_01"
DEVICE = "cuda"  # use "cpu" only if no GPU is available
BATCH_SIZE = 1
NUM_EXAMPLES = 3
DEMO_SCORE_THRESHOLD = 0.25  # same threshold as previous training demos; no automatic lowering


In [ ]:
import importlib.util
import pathlib
import subprocess
import sys
import time
SESSION_START = time.monotonic()
assert "YOUR_USER" not in REPO_URL, "Set REPO_URL first"
assert RUN_NAME and pathlib.Path(RUN_NAME).name == RUN_NAME, "RUN_NAME must be a directory name"
import torch
assert DEVICE in {"cuda", "cpu"}
assert DEVICE != "cuda" or torch.cuda.is_available(), "Enable a Kaggle GPU or set DEVICE=cpu"
print("PyTorch:", torch.__version__, "Inference device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0), "(one GPU is sufficient for inference)")
# Keep Kaggle's CUDA PyTorch. Do not install the local CPU requirements.lock.txt.
missing = [pkg for module, pkg in [("PIL", "pillow>=10,<13"), ("matplotlib", "matplotlib>=3.8,<4"), ("numpy", "numpy>=1.26,<3")] if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
WORK = pathlib.Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)
# Source stays outside /kaggle/working so saved output contains only the run and ZIP.
REPO = pathlib.Path("/kaggle/temp/personal_yolov1_inference_repo")
REPO.parent.mkdir(parents=True, exist_ok=True)
if not (REPO / ".git").is_dir():
    assert not REPO.exists(), f"Non-git path already exists: {REPO}"
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(REPO)], check=True)
else:
    origin = subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip()
    assert origin == REPO_URL, "Existing checkout has a different origin; use a fresh Kaggle session"
subprocess.run(["git", "-C", str(REPO), "sparse-checkout", "init", "--cone"], check=True)
subprocess.run(["git", "-C", str(REPO), "sparse-checkout", "set", "code"], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)
CODE = REPO / "code"
assert (CODE / "inference_demo.py").is_file(), "Push code/inference_demo.py to GitHub before running this notebook"
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Code commit:", COMMIT)
if REPO_REF != COMMIT:
    print("For repeatable runs set REPO_REF to:", COMMIT)
sys.path.insert(0, str(CODE))


In [ ]:
from dataset import find_voc_root
from inference_demo import resolve_checkpoint
INPUT = pathlib.Path("/kaggle/input")
DATA = find_voc_root(DATASET_DIR or INPUT)
CACHE = pathlib.Path("/kaggle/temp/yolov1_inference_weights")
checkpoint_source = CHECKPOINT_PATH
if checkpoint_source and not pathlib.Path(checkpoint_source).is_absolute():
    checkpoint_source = str(INPUT / checkpoint_source)
CHECKPOINT, ORIGIN = resolve_checkpoint(checkpoint_source, INPUT, CACHE)
OUT = WORK / RUN_NAME
ZIP = WORK / f"{RUN_NAME}.zip"
assert not OUT.exists(), "Choose a new RUN_NAME or restart the Kaggle session"
print("Dataset:", DATA)
print("Checkpoint:", CHECKPOINT, "Source:", ORIGIN)
print("Output:", ZIP)


In [ ]:
# Fresh inference/evaluation on ALL official validation images (no --limit).
# Metrics at eval threshold 0.001 stay separate from demo filtering at 0.25.
from inference_demo import run_demo
RESULT = run_demo(
    data=DATA, checkpoint=CHECKPOINT, output=OUT, zip_path=ZIP,
    device=DEVICE, batch_size=BATCH_SIZE, count=NUM_EXAMPLES,
    demo_threshold=DEMO_SCORE_THRESHOLD, checkpoint_source=ORIGIN,
)
assert RESULT["status"] == "completed"
print("ZIP bytes:", ZIP.stat().st_size)


In [ ]:
import json
from IPython.display import FileLink, Image as NotebookImage, display
metrics = json.loads((OUT / "metrics.json").read_text())
operating = json.loads((OUT / "operating_point_metrics.json").read_text())
selected = json.loads((OUT / "selected_examples.json").read_text())
print(f"Validation images: {metrics['images']}; mAP@0.5: {100 * metrics['map50']:.2f}%")
print(f"At score >= {DEMO_SCORE_THRESHOLD}: precision={operating['precision']:.3f}, "
      f"recall={operating['recall']:.3f}, F1={operating['f1']:.3f}")
print("Illustrative image IDs:", [row["image_id"] for row in selected["examples"]])
print("Selection uses GT after inference. These three images are not an unbiased evaluation sample.")
if selected["actual_count"] < NUM_EXAMPLES:
    print("Fewer qualifying images exist at this score threshold; it was not changed automatically.")
for row in selected["examples"]:
    display(NotebookImage(filename=str(OUT / "examples" / f"{row['image_id']}_comparison.jpg"), width=1400))
display(NotebookImage(filename=str(OUT / "per_class_ap.png"), width=900))
display(FileLink(str(ZIP)))
# Save Version -> Save & Run All, then download the ZIP from the Output tab.


## ZIP contents

- `metrics.json`, `per_class_ap.csv`: VOC2007 11-point mAP/AP over the full val split.
- `operating_point_metrics.json`: micro precision/recall/F1, TP/FP/FN and mean TP IoU at the selected display threshold, over **all val images**.
- `threshold_metrics.csv`: operating points at score 0.05/0.10/0.25/0.50 (plus any custom display threshold).
- `selected_examples.json/csv`, `examples/selected_detections.csv`: per-example scores, classes, IoUs and match status.
- `examples/*_comparison.jpg`: original / ground truth / prediction; `contact_sheet.jpg` groups the three illustrations.
- `examples/*_input.jpg`, `*_predictions.png`, `per_class_ap.png/pdf`, `summary.txt`.
- `predictions.json`, `per_image_metrics.csv`, `run_manifest.json`, code snapshot and artifact checksums.

No checkpoint is copied into the demo output ZIP. Use scores/IoUs as measured; do not interpret selected examples as the model's overall accuracy.
